In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv('adult.csv')
df.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   age              48842 non-null  int64 
 1   workclass        48842 non-null  object
 2   fnlwgt           48842 non-null  int64 
 3   education        48842 non-null  object
 4   educational-num  48842 non-null  int64 
 5   marital-status   48842 non-null  object
 6   occupation       48842 non-null  object
 7   relationship     48842 non-null  object
 8   race             48842 non-null  object
 9   gender           48842 non-null  object
 10  capital-gain     48842 non-null  int64 
 11  capital-loss     48842 non-null  int64 
 12  hours-per-week   48842 non-null  int64 
 13  native-country   48842 non-null  object
 14  income           48842 non-null  object
dtypes: int64(6), object(9)
memory usage: 5.6+ MB


In [4]:
df.isnull().sum()

,0
age,0
workclass,0
fnlwgt,0
education,0
educational-num,0
marital-status,0
occupation,0
relationship,0
race,0
gender,0


In [5]:
df.dtypes

,0
age,int64
workclass,object
fnlwgt,int64
education,object
educational-num,int64
marital-status,object
occupation,object
relationship,object
race,object
gender,object


In [ ]:
df['income'].value_counts()

,count
income,
<=50K,37155
>50K,11687


In [6]:
df.replace('?', np.nan, inplace=True)


df['income'] = (df['income'].str.strip() == '>50K').astype(int)

X = df.drop('income', axis=1)
y = df['income']


numeric_cols   = X.select_dtypes(include='number').columns.tolist()
categorical_cols = X.select_dtypes(exclude='number').columns.tolist()

print(f"Numeric features  : {numeric_cols}")
print(f"Categorical features: {categorical_cols}")
print(f"\nX shape: {X.shape}")
print(f"y distribution:\n{y.value_counts()}")

Numeric features  : ['age', 'fnlwgt', 'educational-num', 'capital-gain', 'capital-loss', 'hours-per-week']
Categorical features: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'gender', 'native-country']

X shape: (48842, 14)
y distribution:
income
0    37155
1    11687
Name: count, dtype: int64


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


# Preprocessing pipelines
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer,   numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

In [8]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=10, random_state=42),
    'SVC':                 SVC(kernel='rbf', C=1.0, random_state=42),
    'KNN':                 KNeighborsClassifier(n_neighbors=5)
}

# Train each model inside a full ML pipeline
results  = {}
pipelines = {}

for name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier',   model)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    acc    = accuracy_score(y_test, y_pred)

    results[name]   = {
        'accuracy': acc,
        'y_pred':   y_pred,
        'report':   classification_report(y_test, y_pred, target_names=['<=50K', '>50K']),
        'cm':       confusion_matrix(y_test, y_pred)
    }
    pipelines[name] = pipe
    print(f" {name}  Accuracy: {acc}")

 Logistic Regression  Accuracy: 0.8523902139420616
 Decision Tree  Accuracy: 0.8607841130105436
 SVC  Accuracy: 0.8595557375371071
 KNN  Accuracy: 0.8336574879721568


In [9]:
# ── Classification Reports ──
for name in models:
    print(f"{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(results[name]['report'])

  Logistic Regression
              precision    recall  f1-score   support

       <=50K       0.88      0.94      0.91      7431
        >50K       0.74      0.59      0.66      2338

    accuracy                           0.85      9769
   macro avg       0.81      0.76      0.78      9769
weighted avg       0.85      0.85      0.85      9769

  Decision Tree
              precision    recall  f1-score   support

       <=50K       0.88      0.94      0.91      7431
        >50K       0.77      0.60      0.67      2338

    accuracy                           0.86      9769
   macro avg       0.83      0.77      0.79      9769
weighted avg       0.85      0.86      0.85      9769

  SVC
              precision    recall  f1-score   support

       <=50K       0.88      0.94      0.91      7431
        >50K       0.77      0.59      0.67      2338

    accuracy                           0.86      9769
   macro avg       0.82      0.77      0.79      9769
weighted avg       0.85      0

In [14]:

import pickle


best_model_name = max(results, key=lambda x: results[x]['accuracy'])


best_model = pipelines[best_model_name]


with open("model.pkl", "wb") as file:
    pickle.dump(best_model, file)

print(f"{best_model_name} saved as model.pkl")

Decision Tree saved as model.pkl


In [13]:
from sklearn.metrics import accuracy_score

# Ensure 'best_model_name' is defined in this scope
best_model_name = max(results, key=lambda x: results[x]['accuracy'])

model_pipeline = pipelines[best_model_name]

y_train_pred = model_pipeline.predict(X_train)
y_test_pred = model_pipeline.predict(X_test)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"Train accuracy: {train_acc:.3f}")
print(f"Test accuracy: {test_acc:.3f}")
print(f"Gap: {train_acc - test_acc:.3f}")

Train accuracy: 0.870
Test accuracy: 0.861
Gap: 0.009
